# Banco de dados e consultas SQL

Esta etapa organiza o armazenamento: todas as camadas (Bronze, Silver e as três Golds) são gravadas num banco de dados relacional, e as perguntas principais do projeto são respondidas com consultas SQL. Assim, qualquer pessoa com acesso ao banco consegue conferir os resultados sem precisar rodar os notebooks de Python.

Usamos o **SQLite** porque ele já vem com o Python, guarda tudo num único arquivo e usa SQL padrão. Num ambiente de empresa, a mesma estrutura iria para um banco com servidor (como o próprio Databricks ou PostgreSQL), e as consultas continuariam praticamente iguais.

## 1. Estrutura do banco

| Tabela | Camada | Conteúdo | Quem acessa |
|---|---|---|---|
| `bronze_clientes` | Bronze | Base original, tudo como texto | Engenharia de dados |
| `silver_clientes` | Silver | Base limpa e tipada, com ID real e feedback | Engenharia de dados |
| `gold_financeiro` | Gold 1 | Perfil + valores, com ID anônimo | Análise |
| `gold_servicos` | Gold 2 | Perfil + serviços, com ID anônimo | Análise |
| `gold_contrato` | Gold 3 | Perfil + contrato e tempo, com ID anônimo | Análise |

As três Golds se ligam pelo `ID_Anonimo`, que é igual nas três para o mesmo cliente.

In [1]:
import os
import sqlite3

import pandas as pd

os.makedirs('../Data/Banco', exist_ok=True)
CAMINHO_BANCO = '../Data/Banco/terra_signal.db'
if os.path.exists(CAMINHO_BANCO):
    os.remove(CAMINHO_BANCO)  # recria do zero para o resultado ser sempre o mesmo

conexao = sqlite3.connect(CAMINHO_BANCO)


def consulta(sql):
    """Executa uma consulta SQL e devolve o resultado como tabela."""
    return pd.read_sql_query(sql, conexao)

### Tabela Silver com regras de integridade

A tabela Silver é criada com o comando `CREATE TABLE` escrito à mão, para deixar claras as regras que o banco deve garantir:

- `PRIMARY KEY` no ID: impede cliente duplicado.
- `NOT NULL`: impede campos obrigatórios vazios.
- `CHECK`: impede valores impossíveis, como mensalidade negativa ou tempo de assinatura maior que 72 meses.

Se algum dado ruim tentar entrar, o banco recusa a gravação. É uma segunda barreira de qualidade, além da limpeza feita em Python.

In [2]:
conexao.execute('''
CREATE TABLE silver_clientes (
    ID_Cliente              TEXT PRIMARY KEY,
    Genero                  TEXT NOT NULL CHECK (Genero IN ('Masculino', 'Feminino')),
    Idoso                   INTEGER NOT NULL CHECK (Idoso IN (0, 1)),
    Conjuge                 INTEGER NOT NULL CHECK (Conjuge IN (0, 1)),
    Dependentes             INTEGER NOT NULL CHECK (Dependentes IN (0, 1)),
    Tempo_Assinatura        INTEGER NOT NULL CHECK (Tempo_Assinatura BETWEEN 0 AND 72),
    Servico_Telefonia       INTEGER NOT NULL CHECK (Servico_Telefonia IN (0, 1)),
    Varias_Linhas           TEXT NOT NULL,
    Servico_Internet        TEXT NOT NULL,
    Seguranca_Online        TEXT NOT NULL,
    Backup_Online           TEXT NOT NULL,
    Protecao_Dispositivos   TEXT NOT NULL,
    Suporte_Tecnico         TEXT NOT NULL,
    Streaming_TV            TEXT NOT NULL,
    Streaming_Filmes        TEXT NOT NULL,
    Contrato                TEXT NOT NULL CHECK (Contrato IN ('Mensal', 'Um ano', 'Dois anos')),
    Faturamento_Eletronico  INTEGER NOT NULL CHECK (Faturamento_Eletronico IN (0, 1)),
    Forma_Pagamento         TEXT NOT NULL,
    Valores_Mensais         REAL NOT NULL CHECK (Valores_Mensais > 0),
    Valores_Totais          REAL NOT NULL CHECK (Valores_Totais >= 0),
    Status_Cancelamento     INTEGER NOT NULL CHECK (Status_Cancelamento IN (0, 1)),
    Feedback_Cliente        TEXT,
    Renda_Mensal            REAL NOT NULL CHECK (Renda_Mensal > 0),
    Tempo_Estimado          INTEGER NOT NULL,
    Total_Estimado          INTEGER NOT NULL
)
''')
print('Tabela silver_clientes criada')

Tabela silver_clientes criada


## 2. Carga das camadas

In [3]:
bronze = pd.read_csv('../Data/Bronze/tabela_bronze.csv', dtype=str, keep_default_na=False)
silver = pd.read_csv('../Data/Silver/tabela_silver.csv')
gold_financeiro = pd.read_csv('../Data/Gold/gold_cliente_financeiro.csv')
gold_servicos = pd.read_csv('../Data/Gold/gold_cliente_servicos.csv')
gold_contrato = pd.read_csv('../Data/Gold/gold_cliente_contrato.csv')

# Bronze: guardada como veio, tudo texto
bronze.to_sql('bronze_clientes', conexao, index=False)

# Silver: entra na tabela com as regras de integridade criada acima
silver.to_sql('silver_clientes', conexao, index=False, if_exists='append')

# Golds: cada uma com índice único no ID anônimo
for nome, tabela in [('gold_financeiro', gold_financeiro), ('gold_servicos', gold_servicos),
                     ('gold_contrato', gold_contrato)]:
    tabela.to_sql(nome, conexao, index=False)
    conexao.execute(f'CREATE UNIQUE INDEX idx_{nome}_id ON {nome} (ID_Anonimo)')
conexao.commit()

consulta('''
SELECT 'bronze_clientes' AS tabela, COUNT(*) AS linhas FROM bronze_clientes
UNION ALL SELECT 'silver_clientes', COUNT(*) FROM silver_clientes
UNION ALL SELECT 'gold_financeiro', COUNT(*) FROM gold_financeiro
UNION ALL SELECT 'gold_servicos',   COUNT(*) FROM gold_servicos
UNION ALL SELECT 'gold_contrato',   COUNT(*) FROM gold_contrato
''')

,tabela,linhas
0,bronze_clientes,5634
1,silver_clientes,5633
2,gold_financeiro,5633
3,gold_servicos,5633
4,gold_contrato,5633


A Bronze tem 5.634 linhas (a base original) e as demais têm 5.633, porque a Silver removeu 1 cliente sem tempo e sem total.

### Teste das regras de integridade

Tentamos inserir um cliente com mensalidade negativa. O banco deve recusar.

In [4]:
try:
    conexao.execute('''
        INSERT INTO silver_clientes VALUES ('0000-TESTE', 'Masculino', 0, 0, 0, 10, 1, 'Não', 'DSL',
        'Não', 'Não', 'Não', 'Não', 'Não', 'Não', 'Mensal', 1, 'Cheque eletrônico', -50, 0, 0, NULL, 3000, 0, 0)
    ''')
    print('O registro foi aceito (a regra falhou)')
except sqlite3.IntegrityError as erro:
    print('Registro recusado pelo banco:', erro)

Registro recusado pelo banco: CHECK constraint failed: Valores_Mensais > 0


## 3. Controle de acesso

O SQLite não tem usuários e senhas, então simulamos o controle de acesso com **visões** (`VIEW`). Uma visão é uma consulta salva que se comporta como uma tabela. A ideia é que o time de análise só enxergue as visões, nunca as tabelas Bronze e Silver, que têm o ID real e o texto do feedback.

Num banco com servidor, isso seria feito com permissões, como vimos em aula:

```sql
CREATE ROLE analista;
GRANT SELECT ON vw_carteira_analise TO analista;
REVOKE ALL ON silver_clientes FROM analista;
```

In [5]:
conexao.executescript('''
-- Visão de análise: junta as três Golds pelo ID anônimo, sem nenhum dado identificável
CREATE VIEW vw_carteira_analise AS
SELECT c.ID_Anonimo, c.Genero, c.Idoso, c.Conjuge, c.Dependentes,
       c.Contrato, c.Forma_Pagamento, c.Faturamento_Eletronico, c.Tempo_Assinatura, c.Faixa_Tempo,
       s.Servico_Internet, s.Seguranca_Online, s.Suporte_Tecnico, s.Qtd_Adicionais, s.Perfil_Pacote,
       f.Valores_Mensais, f.Faixa_Mensalidade, c.Grupo_Risco, c.Status_Cancelamento
FROM gold_contrato c
JOIN gold_servicos s   ON s.ID_Anonimo = c.ID_Anonimo
JOIN gold_financeiro f ON f.ID_Anonimo = c.ID_Anonimo;

-- Visão agregada: só totais por contrato, sem nenhum cliente individual
CREATE VIEW vw_resumo_contrato AS
SELECT Contrato,
       COUNT(*) AS clientes,
       SUM(Status_Cancelamento) AS cancelaram,
       ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa_cancelamento
FROM gold_contrato
GROUP BY Contrato;
''')

consulta("SELECT name AS objeto, type AS tipo FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name")

,objeto,tipo
0,bronze_clientes,table
1,gold_contrato,table
2,gold_financeiro,table
3,gold_servicos,table
4,silver_clientes,table
5,vw_carteira_analise,view
6,vw_resumo_contrato,view


Daqui em diante, todas as consultas usam só a visão `vw_carteira_analise`, como faria alguém do time de análise.

## 4. Consultas que respondem às perguntas do projeto

**Pergunta central:** quais características da carteira de clientes estão associadas ao cancelamento, e quais grupos concentram esse risco?

### 4.1 Qual é a taxa geral de cancelamento?

In [6]:
consulta('''
SELECT COUNT(*) AS clientes,
       SUM(Status_Cancelamento) AS cancelaram,
       ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa_cancelamento
FROM vw_carteira_analise
''')

,clientes,cancelaram,taxa_cancelamento
0,5633,1503,26.7


### 4.2 O tipo de contrato muda o cancelamento? (Gold 3)

A coluna `participacao_nos_cancelamentos` usa uma função de janela (`SUM(...) OVER ()`) para dividir os cancelamentos de cada contrato pelo total de cancelamentos da base.

In [7]:
consulta('''
SELECT Contrato,
       COUNT(*) AS clientes,
       ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa_cancelamento,
       ROUND(100.0 * SUM(Status_Cancelamento) / SUM(SUM(Status_Cancelamento)) OVER (), 1) AS participacao_nos_cancelamentos
FROM vw_carteira_analise
GROUP BY Contrato
ORDER BY taxa_cancelamento DESC
''')

,Contrato,clientes,taxa_cancelamento,participacao_nos_cancelamentos
0,Mensal,3099,42.7,88.0
1,Um ano,1187,11.5,9.1
2,Dois anos,1347,3.3,2.9


### 4.3 Como o tempo de permanência se relaciona com o cancelamento? (Gold 3)

In [8]:
consulta('''
SELECT CASE
           WHEN Tempo_Assinatura <= 12 THEN '1. Primeiro ano'
           WHEN Tempo_Assinatura <= 24 THEN '2. Segundo ano'
           WHEN Tempo_Assinatura <= 48 THEN '3. Terceiro e quarto ano'
           ELSE '4. Mais de 4 anos'
       END AS periodo,
       COUNT(*) AS clientes,
       ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa_cancelamento
FROM vw_carteira_analise
GROUP BY periodo
ORDER BY periodo
''')

,periodo,clientes,taxa_cancelamento
0,1. Primeiro ano,1756,46.8
1,2. Segundo ano,808,28.6
2,3. Terceiro e quarto ano,1291,21.8
3,4. Mais de 4 anos,1778,9.5


### 4.4 Quais serviços se relacionam com o cancelamento? (Gold 2)

Entre os clientes com internet, comparamos o tipo de internet e a presença de segurança online ou suporte técnico.

In [9]:
consulta('''
SELECT Servico_Internet,
       CASE WHEN Seguranca_Online = 'Sim' OR Suporte_Tecnico = 'Sim'
            THEN 'Com segurança ou suporte' ELSE 'Sem segurança e sem suporte' END AS protecao,
       COUNT(*) AS clientes,
       ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa_cancelamento
FROM vw_carteira_analise
WHERE Servico_Internet <> 'Sem serviço de internet'
GROUP BY Servico_Internet, protecao
ORDER BY taxa_cancelamento DESC
''')

,Servico_Internet,protecao,clientes,taxa_cancelamento
0,Fibra óptica,Sem segurança e sem suporte,1423,55.1
1,DSL,Sem segurança e sem suporte,635,35.7
2,Fibra óptica,Com segurança ou suporte,1053,24.4
3,DSL,Com segurança ou suporte,1289,11.3


### 4.5 A forma de pagamento influencia o cancelamento? (Gold 3)

In [10]:
consulta('''
SELECT Forma_Pagamento,
       COUNT(*) AS clientes,
       ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa_cancelamento
FROM vw_carteira_analise
GROUP BY Forma_Pagamento
ORDER BY taxa_cancelamento DESC
''')

,Forma_Pagamento,clientes,taxa_cancelamento
0,Cheque eletrônico,1888,45.1
1,Cheque pelo correio,1283,19.4
2,Transferência bancária automática,1228,17.1
3,Cartão de crédito automático,1234,15.6


### 4.6 A mensalidade e o perfil do cliente se relacionam com o cancelamento? (Gold 1)

In [11]:
consulta('''
SELECT CASE WHEN Status_Cancelamento = 1 THEN 'Cancelou' ELSE 'Ficou' END AS situacao,
       COUNT(*) AS clientes,
       ROUND(AVG(Valores_Mensais), 2) AS mensalidade_media,
       ROUND(AVG(Tempo_Assinatura), 1) AS tempo_medio_meses,
       ROUND(100.0 * AVG(Idoso), 1) AS pct_idosos,
       ROUND(100.0 * AVG(CASE WHEN Contrato = 'Mensal' THEN 1 ELSE 0 END), 1) AS pct_contrato_mensal
FROM vw_carteira_analise
GROUP BY situacao
''')

,situacao,clientes,mensalidade_media,tempo_medio_meses,pct_idosos,pct_contrato_mensal
0,Cancelou,1503,74.61,18.4,26.1,88.0
1,Ficou,4130,60.96,37.4,13.0,43.0


### 4.7 Qual grupo concentra o risco? (resposta à pergunta central)

In [12]:
consulta('''
SELECT CASE WHEN Grupo_Risco = 1 THEN 'Mensal + fibra + cheque eletrônico' ELSE 'Demais clientes' END AS grupo,
       COUNT(*) AS clientes,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_da_carteira,
       ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa_cancelamento,
       ROUND(100.0 * SUM(Status_Cancelamento) / SUM(SUM(Status_Cancelamento)) OVER (), 1) AS pct_dos_cancelamentos
FROM vw_carteira_analise
GROUP BY grupo
''')

,grupo,clientes,pct_da_carteira,taxa_cancelamento,pct_dos_cancelamentos
0,Demais clientes,4595,81.6,19.2,58.6
1,Mensal + fibra + cheque eletrônico,1038,18.4,59.9,41.4


### 4.8 Visão agregada para gestão

Exemplo de uso da visão `vw_resumo_contrato`, que pode ser compartilhada com áreas que só precisam de números totais.

In [13]:
consulta('SELECT * FROM vw_resumo_contrato ORDER BY taxa_cancelamento DESC')

,Contrato,clientes,cancelaram,taxa_cancelamento
0,Mensal,3099,1322,42.7
1,Um ano,1187,137,11.5
2,Dois anos,1347,44,3.3


## 5. Conferência com o Python

Para garantir que o banco e os notebooks contam a mesma história, comparamos a taxa por contrato calculada em SQL com a calculada em pandas na Gold 3.

In [14]:
sql = consulta('SELECT Contrato, ROUND(100.0 * AVG(Status_Cancelamento), 1) AS taxa FROM vw_carteira_analise GROUP BY Contrato').set_index('Contrato')['taxa']
python = (gold_contrato.groupby('Contrato')['Status_Cancelamento'].mean() * 100).round(1)
pd.DataFrame({'SQL': sql, 'Python': python, 'Iguais': sql == python})

,SQL,Python,Iguais
Contrato,,,
Dois anos,3.3,3.3,True
Mensal,42.7,42.7,True
Um ano,11.5,11.5,True


In [15]:
conexao.close()
print('Banco salvo em', CAMINHO_BANCO)

Banco salvo em ../Data/Banco/terra_signal.db


## Resumo

- O banco guarda as cinco tabelas do projeto num único arquivo (`Data/Banco/terra_signal.db`), que pode ser aberto em qualquer ferramenta compatível com SQLite (DB Browser for SQLite, DBeaver, extensão do VS Code).
- A tabela Silver tem regras de integridade que impedem dados inválidos.
- O acesso de análise passa por visões sem dados identificáveis.
- As consultas confirmam os resultados das Golds: contrato mensal, primeiro ano, fibra sem proteção e cheque eletrônico são os pontos de maior cancelamento, e o grupo mensal + fibra + cheque eletrônico concentra 41% dos cancelamentos com 18% dos clientes.